# فصل 13: پایگاه‌داده‌ی مکانی (PostgreSQL و PostGIS)

کد این نوت‌بوک عیناً از فصل 13 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۱۳-۸) اتصال امن پایتون به PostgreSQL


In [ ]:
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()          # read DB_* from a .env file (never committed)

user, pw = os.getenv("DB_USER"), os.getenv("DB_PASSWORD")
host, port = os.getenv("DB_HOST"), os.getenv("DB_PORT")
name = os.getenv("DB_NAME")
url = f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{name}"
engine = create_engine(url, pool_pre_ping=True)

# SAFE: named parameter (NOT an f-string inside the query)
with engine.connect() as conn:
    rows = conn.execute(
        text("SELECT name, capacity FROM hospitals WHERE city = :city"),
        {"city": user_input})
    for r in rows:
        print(r.name, r.capacity)

## ۱۳-۹) GeoPandas و PostGIS: جریان واقعی کار


In [ ]:
import geopandas as gpd
from database import engine

hospitals = gpd.read_postgis("SELECT id, name, geom FROM hospitals",
                             engine, geom_col="geom")

# WRONG: distance in EPSG:4326 is in DEGREES, not metres!
# RIGHT: convert to a metric CRS (UTM) first
hospitals_utm = hospitals.to_crs(epsg=32639)
hospitals_utm["buffer_2km"] = hospitals_utm.geometry.buffer(2000)   # metres